In [ ]:
# Effect of dessexualizacao specifically — the article's core claim
print("="*60)
print("  Dessexualização como preditor isolado do ENDURECIMENTO")
print("="*60)

from scipy.stats import spearmanr

for ind in ['dessexualizacao', 'serialidade', 'inscricao_estatal', 'monocromatizacao']:
    rho, p = spearmanr(df[ind], df['purificacao_composto'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {ind:<30s} ρ = {rho:.3f}  p = {p:.4f} {sig}")

# Scatter: dessexualizacao vs composite
fig, ax = plt.subplots(figsize=(8, 5))
palette = {'fundacional': '#e74c3c', 'normativo': '#3498db', 'militar': '#2c3e50'}
for regime in ['fundacional', 'normativo', 'militar']:
    sub = df[df.regime_iconocratico == regime]
    ax.scatter(sub['dessexualizacao'] + np.random.normal(0, 0.08, len(sub)),
              sub['purificacao_composto'], alpha=0.6, label=regime.capitalize(),
              color=palette[regime], s=50)
ax.set_xlabel('Dessexualização (0-3)')
ax.set_ylabel('Score composto ENDURECIMENTO')
ax.set_title('Dessexualização vs. ENDURECIMENTO por regime')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig_06_dessex_vs_endurec.png', dpi=150, bbox_inches='tight')
plt.show()

# 03 — Preditores do ENDURECIMENTO: Regressão Ordinal (Cap. 6.3)

Quais variáveis (regime, período, país, suporte) predizem o índice de purificação?

**Modelo:** Regressão logística ordinal (proportional odds)

**VD:** Score composto de ENDURECIMENTO (discretizado em quartis)

**VIs:** regime_iconocratico, country, period_norm, medium_norm

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

df = pd.read_csv('../data/processed/corpus_dataset.csv')

# Discretize composite into quartile levels for ordinal regression
df['endurec_level'] = pd.qcut(df['purificacao_composto'], q=4, labels=['Q1_baixo','Q2','Q3','Q4_alto'])
print("Distribuição dos níveis de ENDURECIMENTO:")
print(df['endurec_level'].value_counts().sort_index())

# Create dummies for predictors
df['regime_num'] = df['regime_iconocratico'].map({'fundacional': 0, 'normativo': 1, 'militar': 2})

# OLS on continuous composite as baseline
print("\n" + "="*60)
print("  OLS: ENDURECIMENTO ~ regime (baseline)")
print("="*60)
X = pd.get_dummies(df['regime_iconocratico'], drop_first=True, dtype=float)
X = sm.add_constant(X)
y = df['purificacao_composto']
model_ols = sm.OLS(y, X).fit()
print(model_ols.summary2().tables[1].to_string())

In [ ]:
# Ordinal regression: ENDURECIMENTO level ~ regime + medium
print("="*60)
print("  Ordinal Logistic Regression: ENDURECIMENTO ~ regime + medium")
print("="*60)

# Encode ordinal DV
df['endurec_ord'] = df['endurec_level'].cat.codes

# Build predictor matrix
predictors = pd.get_dummies(df[['regime_iconocratico']], drop_first=True, dtype=float)

# Add medium if enough categories
medium_dummies = pd.get_dummies(df['medium_norm'], prefix='med', drop_first=True, dtype=float)
# Keep only mediums with N >= 5
med_cols = [c for c in medium_dummies.columns if medium_dummies[c].sum() >= 5]
predictors = pd.concat([predictors, medium_dummies[med_cols]], axis=1)

try:
    ord_model = OrderedModel(df['endurec_ord'], predictors, distr='logit')
    ord_result = ord_model.fit(method='bfgs', disp=False)
    print(ord_result.summary())
except Exception as e:
    print(f"Ordinal model failed: {e}")
    print("Falling back to OLS with full predictors:")
    X_full = sm.add_constant(predictors)
    model_full = sm.OLS(df['purificacao_composto'], X_full).fit()
    print(model_full.summary2().tables[1].to_string())
    print(f"\nR² = {model_full.rsquared:.3f}, Adj. R² = {model_full.rsquared_adj:.3f}")